In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [46]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [47]:
df.head()

,text,emotion
0,i can go from feeling so hopeless to so damned...,sadness
1,im grabbing a minute to post i feel greedy wrong,anger
2,i am ever feeling nostalgic about the fireplac...,love
3,i am feeling grouchy,anger
4,ive been feeling a little burdened lately wasn...,sadness


In [48]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [49]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
  emotion_numbers[emo] = i
  i +=1

df['emotion'] = df['emotion'].map(emotion_numbers)

In [50]:
df.head()

,text,emotion
0,i can go from feeling so hopeless to so damned...,0
1,im grabbing a minute to post i feel greedy wrong,1
2,i am ever feeling nostalgic about the fireplac...,2
3,i am feeling grouchy,1
4,ive been feeling a little burdened lately wasn...,0


In [51]:
df['text'] = df['text'].apply(lambda x : x.lower()) # anonymous fun(fun does not hav name)
df.head()

,text,emotion
0,i can go from feeling so hopeless to so damned...,0
1,im grabbing a minute to post i feel greedy wrong,1
2,i am ever feeling nostalgic about the fireplac...,2
3,i am feeling grouchy,1
4,ive been feeling a little burdened lately wasn...,0


In [52]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [53]:
df['text'] = df['text'].apply(remove_punc)

In [54]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [55]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [56]:
import nltk

In [57]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [58]:
nltk.download('punkt') # download_dir='/nltk_data'
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [59]:
stop_words = set(stopwords.words('english'))
print(stop_words)

{'hers', 'some', 'there', 'wasn', 'this', 'by', 'didn', 'he', 'until', 'ain', 'have', 'through', 'weren', "he'd", 'will', 'up', 'how', "he'll", 'theirs', 'haven', 'not', 'or', 'as', "they'll", 'be', 'their', 'no', 'under', 'with', 'having', 'what', "mustn't", 'both', 'herself', "wasn't", 'during', 'his', 'such', 'were', 'needn', 'doesn', "shouldn't", 'these', 'about', "wouldn't", 'then', 'in', 'm', 'any', 's', 'same', 'mightn', 'an', 'too', 'than', "you'd", "you'll", 'below', 'they', 'so', 'whom', "shan't", 'being', 'out', 'shan', 'off', 'those', "she's", 'that', 'her', 'after', 'other', "i've", 'y', "mightn't", 'most', 'own', 'she', 'more', 'isn', 'has', 'himself', 'doing', 'who', 'mustn', 'does', 're', 'further', 'if', 'was', 'is', 'o', 'itself', "he's", 'of', 'on', 'again', 'for', 'when', "aren't", 'between', "it'd", 'it', 'd', 'just', 'why', "we've", "didn't", "she'll", 'hasn', 'had', 'll', 'to', 'won', "doesn't", 'do', 'a', 'before', "she'd", 'above', 'aren', 'and', 'my', 'while',

In [60]:
df.loc[1]['text']

'im grabbing a minute to post i feel greedy wrong'

In [61]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [62]:
df['text'] = df['text'].apply(remove)

In [63]:
df.loc[1]['text']

'im grabbing minute post feel greedy wrong'

In [64]:
df.head()

,text,emotion
0,go feeling hopeless damned hopeful around some...,0
1,im grabbing minute post feel greedy wrong,1
2,ever feeling nostalgic fireplace know still pr...,2
3,feeling grouchy,1
4,ive feeling little burdened lately wasnt sure,0


In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [66]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.765625


In [67]:
pred_bow

array([0, 1, 0, ..., 0, 5, 0], shape=(3200,))

In [68]:

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [69]:
y_pred = nb2_model.predict(X_test_tfidf)

In [70]:
print(accuracy_score(y_test, y_pred))

0.6715625


In [71]:
from sklearn.linear_model import LogisticRegression

In [72]:
logistic_model = LogisticRegression(max_iter=1000)

In [73]:
logistic_model.fit(X_train_tfidf,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [74]:
log_pred = logistic_model.predict(X_test_tfidf)

In [75]:
print(accuracy_score(y_test,log_pred ))

0.86125


In [76]:
# ---- Single-text prediction helpers and interactive prompt ----
# Reuse preprocessing functions defined earlier: remove_punc, remove_numbers, remove_emojis, remove
def preprocess_text(txt):
    t = txt.lower()
    t = remove_punc(t)
    t = remove_numbers(t)
    t = remove_emojis(t)
    t = remove(t)  # remove stopwords using the 'remove' function from earlier cells
    return t

def predict_emotion(text):
    """Return the predicted emotion label for a single text input."""
    cleaned = preprocess_text(text)
    # Vectorize with the trained TF-IDF vectorizer and predict with the trained model
    vec = tfidf_vectorizer.transform([cleaned])
    pred_num = logistic_model.predict(vec)[0]
    # Reverse the emotion_numbers mapping to get the original label string
    rev_map = {v: k for k, v in emotion_numbers.items()}
    return rev_map.get(pred_num, str(pred_num))

# Interactive prompt (works when running the notebook in a terminal-backed kernel)
user_text = input('Enter a short text to predict its emotion: ')
if user_text.strip():
    print('Predicted emotion:', predict_emotion(user_text))
else:
    print('No input provided.')

Predicted emotion: sadness
